# Flash Loans — Where Contracts Go Awry

Notebook 6 built two contracts on the chain we already had: a swap pool and a lending protocol. The lending rule asked the pool what ETH was worth. That is the bug. This notebook is the shove.

First Alice uses **40 ETH she already owns** — two transactions, dump then liquidate. Then a **flash loan** rents her the boulder for one atomic transaction. Same puddle. Same victim. Same 3.18 ETH. She no longer has to be rich *before* lunch.

**Question:** how can a protocol execute every instruction correctly, revert when a step fails, and still lose money when every step succeeds?

**Scope:** a deterministic teaching model, not a recipe for attacking or operating a real protocol. It omits fees, liquidity providers, slippage limits, token transfers, external arbitrage, and real-world legal or economic consequences.

The toy model is ours. The mechanisms are not — see **Sources** at the end.


## Recap

Notebook 6 left us `AMMPool`, `LendingProtocol`, `Loan`, `MedianOracle`, and `PriceReport` — defined inline there, imported here from [`blockchain_lib/contracts.py`](../blockchain_lib/contracts.py). Same classes, same pattern as mempool after notebook 5.

What notebook 6 did *not* do is rob anyone. The 40 ETH boulder stayed a thought experiment: a pebble and a boulder in separate fresh pools, and a median that ignored one liar.

We **import** those classes. We do not reinvent them. First we shove the puddle with a suitcase of cash (two gossiped `Transaction`s). Then we rent the suitcase: `broadcast` and `include` one payload, then run `FlashLoanProvider.execute`. That last split is on purpose. On a real chain, inclusion and execution share a block; here they are two cells so you can see a reverted call still leave a receipt. **Included is not the same as succeeded.**

A perfectly atomic robbery is still a robbery.


In [ ]:
import random

from blockchain_lib.contracts import AMMPool, LendingProtocol
from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Blockchain, Validator


def submit_call(network, chain, origin, tx_id, contract, method, **kwargs):
    """Gossip a contract call, include it, then run it.

    Inclusion is the notary stamp. ``call`` is the form being filled in.
    Same helper as notebook 6 — two of these make two transactions.
    """
    arg_preview = ", ".join(f"{k}={v!r}" for k, v in kwargs.items())
    description = f"{contract.address}.{method}({arg_preview})"
    network.broadcast(Transaction(tx_id, description), origin=origin)
    tx, block = network.include(origin, tx_id, chain)
    result = contract.call(method, **kwargs)
    return result, tx, block


## 1. You can shove the puddle — if you brought your own ETH

No lying courier required. The pool will tell the truth about *itself*, and that truth will be a terrible proxy for "the market."

Alice has **40 ETH of her own**. Same boulder you watched in notebook 6. We dump and liquidate as **two transactions**. Between them the cheap price sits on-chain, in public, the way notebook 5 said a waiting room works.

This is not a flash loan. Alice had to be rich *before* lunch.

> Pause and predict: after paying the $12,000 debt and buying ETH back, how much of Alice's original 40 ETH remains — and did she need to own those 40 ETH the whole time?

In [1]:
# Fresh puddle so the 40 ETH splash matches notebook 6's thin pool.
nodes = ["Node A", "Alice-Node", "Bob-Node", "Farid-Node"]
validators = [
    Validator("Node A", 100),
    Validator("Alice-Node", 80),
    Validator("Bob-Node", 70),
    Validator("Farid-Node", 50),
]
owned_network = Network(nodes, random.Random(7))
owned_chain = Blockchain(validators)

owned_pool = AMMPool("amm-thin.eth", 50.0, 100_000.0)
owned_lending = LendingProtocol("lending-thin.eth", owned_pool)
owned_lending.call(
    "open_loan", borrower="victim", collateral_eth=10.0, debt_usd=12_000.0
)

alice_eth = 40.0
usd_from_dump, dump_tx, dump_block = submit_call(
    owned_network,
    owned_chain,
    "Alice-Node",
    "tx-dump-40eth",
    owned_pool,
    "swap_eth_for_usd",
    eth_in=alice_eth,
)
manipulated_price = owned_pool.spot_price
victim_ratio = owned_lending.collateral_ratio(owned_lending.loans["victim"])
seized, liq_tx, liq_block = submit_call(
    owned_network,
    owned_chain,
    "Alice-Node",
    "tx-liquidate-victim",
    owned_lending,
    "liquidate",
    borrower="victim",
)
usd_for_buyback = usd_from_dump - seized.debt_usd
eth_bought_back = owned_pool.call("swap_usd_for_eth", usd_in=usd_for_buyback)
alice_final_eth = eth_bought_back + seized.collateral_eth
profit_eth = alice_final_eth - alice_eth

print(f"{dump_block.proposer} included {dump_tx.tx_id} in Block #{dump_block.index}")
print(
    f"DUMP: Alice sells {alice_eth:.2f} ETH of her own for ${usd_from_dump:,.2f}; "
    f"AMM price becomes ${manipulated_price:,.2f}/ETH."
)
print(
    f"The cheap puddle is now public. Victim ratio at this price: {victim_ratio:.2f}."
)
print(f"{liq_block.proposer} included {liq_tx.tx_id} in Block #{liq_block.index}")
print(
    f"LIQUIDATE: pay ${seized.debt_usd:,.2f} debt and seize "
    f"{seized.collateral_eth:.2f} ETH."
)
print(
    f"BUY BACK: remaining ${usd_for_buyback:,.2f} buys {eth_bought_back:.2f} ETH."
)
print(
    f"Alice started with {alice_eth:.2f} ETH, ends with {alice_final_eth:.2f} ETH, "
    f"profit {profit_eth:.2f} ETH."
)
print("She had to own the 40 ETH already. The bug is the price source, not the financing.")


Alice-Node included tx-dump-40eth in Block #1
DUMP: Alice sells 40.00 ETH of her own for $44,444.44; AMM price becomes $617.28/ETH.
The cheap puddle is now public. Victim ratio at this price: 0.51.
Alice-Node included tx-liquidate-victim in Block #2
LIQUIDATE: pay $12,000.00 debt and seize 10.00 ETH.
BUY BACK: remaining $32,444.44 buys 33.18 ETH.
Alice started with 40.00 ETH, ends with 43.18 ETH, profit 3.18 ETH.
She had to own the 40 ETH already. The bug is the price source, not the financing.


**Read the result:** the dump produces $44,444.44 and makes the victim look unsafe at a 0.51 ratio. Paying the $12,000 debt leaves $32,444.44 for the reverse swap, which buys back about 33.18 ETH. Add the seized 10 ETH collateral, subtract the 40 Alice started with, and about **3.18 ETH** remains as profit.

Every step followed the rules. The rules asked the puddle for the price of the ocean. The dump and the liquidation were *two* transactions, so the cheap price sat in the public waiting room between them. In this toy nobody snipes it. On a real network, somebody might.

Two things before we rent the suitcase:

1. **The bug is the oracle choice.** Alice did not break arithmetic. She fed the lending contract a local ratio and it believed her.
2. **The financing is still a suitcase of cash.** Alice needed 40 ETH sitting around. A flash loan is what happens when you rent the suitcase for one elevator ride, and dump + liquidate + repay become *one* atomic transaction.

## 2. The same attack, rented

You just watched Alice do this with money she already had. Inside **one atomic transaction**, a flash loan lets her: (1) borrow ETH, (2) dump it into a small ETH/USD pool, (3) liquidate a victim whom the distorted price makes look unsafe, (4) buy the borrowed ETH back, (5) repay the principal, and (6) keep the seized collateral. Flash loans do not create the bug. They just mean the attacker does not need a suitcase of cash first.

| Stage | What changes | Why it matters |
| --- | --- | --- |
| Borrow | attacker temporarily receives ETH | capital is available only for this transaction |
| Dump | AMM spot price falls | the lending rule sees a misleading input |
| Liquidate | attacker pays debt and receives collateral | the vulnerable rule follows its price source |
| Buy back and repay | borrowed principal returns to the lender | any remaining ETH belongs to the attacker |

> Pause and predict: after paying the $12,000 debt and reversing the dump, how much ETH remains once the 40 ETH principal is repaid? (You have seen this movie. The ending should look familiar.)


## 3. Flash loans and the heist

A flash-loan provider lends ETH only if the principal is back by the end of the same transaction: enormous buying power with a same-block return policy. This toy provider charges no fee. It snapshots everything it touches and restores that snapshot if anything fails. Only ETH left after repayment counts as attacker profit.

The provider has 1,000 ETH; the attacker borrows 40. The pool starts at 50 ETH and $100,000 — the same puddle as notebook 6.

> Pause and predict: if SketchyGuy-Node originates the attack, can a peer who has not received it include it?


In [5]:
import random
from dataclasses import dataclass

from blockchain_lib.contracts import (
    AMMPool,
    LendingProtocol,
    Loan,
    MedianOracle,
    PositionNotLiquidatableError,
    PriceReport,
)
from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Block, Blockchain, Validator


class InsufficientRepaymentError(Exception):
    """Raised when a flash-loan action cannot return its borrowed principal."""


@dataclass(frozen=True)
class WorldSnapshot:
    """A pre-action copy of every modeled object a flash loan might mutate."""

    pool_eth: float
    pool_usd: float
    loans: dict[str, Loan]
    provider_eth: float


@dataclass(frozen=True)
class AttackTrace:
    """A structured record of what a flash-loan attack actually did."""

    borrowed_eth: float
    usd_from_dump: float
    manipulated_price: float
    victim_ratio: float
    debt_paid_usd: float
    collateral_seized: float
    eth_bought_back: float
    eth_before_repayment: float
    principal_repaid: float


class FlashLoanProvider:
    """Lends ETH that must be repaid before the same call returns.

    Snapshots every modeled object the action touches beforehand and
    restores that snapshot if the action raises or fails to repay --
    modelling atomic all-or-nothing execution without a real EVM.
    """

    def __init__(self, eth_available: float) -> None:
        if eth_available <= 0:
            raise ValueError("Flash-loan liquidity must be positive.")
        self.eth_available = eth_available

    def execute(self, amount_eth, action, pool, protocol):
        """Lend ``amount_eth``, run ``action``, then commit or roll back."""
        if amount_eth <= 0 or amount_eth > self.eth_available:
            raise ValueError("Flash-loan amount is unavailable.")
        snapshot = WorldSnapshot(
            pool.eth_reserve,
            pool.usd_reserve,
            dict(protocol.loans),
            self.eth_available,
        )
        self.eth_available -= amount_eth
        try:
            eth_before_repayment, trace = action(amount_eth)
            if eth_before_repayment < amount_eth:
                raise InsufficientRepaymentError(
                    f"Only {eth_before_repayment:.4f} ETH available to repay "
                    f"{amount_eth:.4f} ETH."
                )
            self.eth_available += amount_eth
            return eth_before_repayment - amount_eth, trace
        except Exception:
            pool.eth_reserve = snapshot.pool_eth
            pool.usd_reserve = snapshot.pool_usd
            protocol.loans = dict(snapshot.loans)
            self.eth_available = snapshot.provider_eth
            raise


def run_flash_attack(
    amount_eth: float, pool: AMMPool, protocol: LendingProtocol, victim: str
) -> tuple[float, AttackTrace]:
    """Borrow, dump, liquidate, buy back, and report the resulting trace."""
    print(f"STEP 1 — BORROW: {amount_eth:.2f} ETH arrives temporarily.")
    usd_from_dump = pool.swap_eth_for_usd(amount_eth)
    manipulated_price = pool.spot_price
    print(
        f"STEP 2 — DUMP: sell {amount_eth:.2f} ETH for ${usd_from_dump:,.2f}; "
        f"AMM price becomes ${manipulated_price:,.2f}/ETH."
    )
    victim_loan = protocol.loans[victim]
    victim_ratio = protocol.collateral_ratio(victim_loan)
    seized_loan = protocol.liquidate(victim)
    debt_paid_usd = seized_loan.debt_usd
    collateral_seized = seized_loan.collateral_eth
    print(
        f"STEP 3 — LIQUIDATE: victim ratio is {victim_ratio:.2f}; "
        f"pay ${debt_paid_usd:,.2f} debt and seize {collateral_seized:.2f} ETH."
    )
    usd_for_buyback = usd_from_dump - debt_paid_usd
    if usd_for_buyback <= 0:
        raise InsufficientRepaymentError("Liquidation leaves no USD for the buyback.")
    eth_bought_back = pool.swap_usd_for_eth(usd_for_buyback)
    eth_before_repayment = eth_bought_back + collateral_seized
    print(
        f"STEP 4 — BUY BACK: remaining ${usd_for_buyback:,.2f} buys back "
        f"{eth_bought_back:.2f} ETH; attacker holds {eth_before_repayment:.2f} ETH."
    )
    print(f"STEP 5 — REPAY: return {amount_eth:.2f} ETH principal to the provider.")
    trace = AttackTrace(
        borrowed_eth=amount_eth,
        usd_from_dump=usd_from_dump,
        manipulated_price=manipulated_price,
        victim_ratio=victim_ratio,
        debt_paid_usd=debt_paid_usd,
        collateral_seized=collateral_seized,
        eth_bought_back=eth_bought_back,
        eth_before_repayment=eth_before_repayment,
        principal_repaid=amount_eth,
    )
    return eth_before_repayment, trace


@dataclass(frozen=True)
class TransactionReceipt:
    """Inclusion records the attempt; status says whether state changes survived."""

    transaction: Transaction
    block: Block
    status: str
    gas_used: int
    state_effect: str


### Submit the heist as a `Transaction`

Same classes as notebook 5, same two calls: `broadcast`, then `include`. A node that missed the gossip cannot propose the attack, no matter how profitable it looks. The difference from the owned-capital dump above: the payload is now *one* transaction, not dump-then-liquidate.

The next cell only stamps the payload onto the chain. The cell after that is execution: borrow, dump, liquidate, repay. If you squint and think "wait, that isn't atomic," you are looking at the teaching split, not a hole in the model. The snapshot inside `execute` is what rewinds the puddle. The block is what remembers that we tried.


In [7]:
nodes = ["Node A", "SketchyGuy-Node", "Emma-Node", "Farid-Node"]
validators = [
    Validator("Node A", 100),
    Validator("SketchyGuy-Node", 80),
    Validator("Emma-Node", 70),
    Validator("Farid-Node", 50),
]
network = Network(nodes, random.Random(7))
chain = Blockchain(validators)
receipts: list[TransactionReceipt] = []

attack_tx = Transaction("tx-flash-attack", "Flash-loan oracle manipulation")
network.broadcast(attack_tx, origin="SketchyGuy-Node")

print("Each node has its own mempool after the attack is gossiped:")
for node in nodes:
    print(f"  {node}: {network.mempool_ids(node)}")

missing = [name for name in nodes if network.get(name, attack_tx.tx_id) is None]
if missing:
    try:
        network.include(missing[0], attack_tx.tx_id, chain)
    except ValueError as error:
        print(f"\n{error}")

included_attack, attack_block = network.include(
    "SketchyGuy-Node", attack_tx.tx_id, chain
)
print(
    f"\n{attack_block.proposer} included {included_attack.tx_id} "
    f"in Block #{attack_block.index}."
)
print(
    "Mempools after inclusion: "
    + ", ".join(f"{name}={network.mempool_ids(name)}" for name in nodes)
)


Each node has its own mempool after the attack is gossiped:
  Node A: ['tx-flash-attack']
  SketchyGuy-Node: ['tx-flash-attack']
  Emma-Node: ['tx-flash-attack']
  Farid-Node: []

Farid-Node cannot include tx-flash-attack: it is not in their local mempool.

SketchyGuy-Node included tx-flash-attack in Block #1.
Mempools after inclusion: Node A=[], SketchyGuy-Node=[], Emma-Node=[], Farid-Node=[]


In [8]:
attack_pool = AMMPool("amm.eth", 50.0, 100_000.0)
attack_protocol = LendingProtocol("lending.eth", attack_pool)
attack_protocol.open_loan("victim", 10.0, 12_000.0)
provider = FlashLoanProvider(1_000.0)

profit_eth, attack_trace = provider.execute(
    40.0,
    lambda borrowed_eth: run_flash_attack(
        borrowed_eth, attack_pool, attack_protocol, "victim"
    ),
    attack_pool,
    attack_protocol,
)
assert attack_trace.eth_before_repayment == (
    attack_trace.principal_repaid + profit_eth
)
assert provider.eth_available == 1_000.0

print("Attack transaction: COMMITTED")
print(f"Provider liquidity after repayment: {provider.eth_available:,.2f} ETH")
print(f"Attacker profit: {profit_eth:.2f} ETH")
print("Same 3.18 ETH as the owned-capital dump. Alice did not need to own the 40 ETH first.")

receipts.append(
    TransactionReceipt(
        included_attack,
        attack_block,
        "SUCCESS",
        310_000,
        "Attack state committed",
    )
)


STEP 1 — BORROW: 40.00 ETH arrives temporarily.
STEP 2 — DUMP: sell 40.00 ETH for $44,444.44; AMM price becomes $617.28/ETH.
STEP 3 — LIQUIDATE: victim ratio is 0.51; pay $12,000.00 debt and seize 10.00 ETH.
STEP 4 — BUY BACK: remaining $32,444.44 buys back 33.18 ETH; attacker holds 43.18 ETH.
STEP 5 — REPAY: return 40.00 ETH principal to the provider.
Attack transaction: COMMITTED
Provider liquidity after repayment: 1,000.00 ETH
Attacker profit: 3.18 ETH
Same 3.18 ETH as the owned-capital dump. Alice did not need to own the 40 ETH first.


**Read the result:** the dump produces $44,444.44 and makes the victim look unsafe at a 0.51 ratio. Paying the $12,000 debt leaves $32,444.44 for the reverse swap, which buys back about 33.18 ETH. Add the seized 10 ETH collateral, repay 40, and about **3.18 ETH** remains as profit. The provider is back at exactly 1,000 ETH.

That is the owned-capital dump with the suitcase rented instead of owned. Every step followed the rules. The rules asked the puddle for the price of the ocean. SketchyGuy-Node could `include` the payload only because `broadcast` had already put it in that node's mempool.


## 4. When one step fails, the whole transaction rewinds

Change only the victim: Victim2 has 40 ETH against the same $12,000 debt. Even after the 40 ETH dump, that position should stay healthy. The provider will still lend, the AMM will still be touched, and then liquidation will raise.

Same `Transaction` path as before: `broadcast`, then `include`, **then** `execute`. The block is appended before liquidation fails. That is the point. The receipt will say `REVERTED`; the chain will still be longer by one. Atomicity refuses to leave a half-finished *world* around. It does not unwrite the attempt.

> Pause and predict: after liquidation fails, which values should look exactly as they did before the flash loan?


In [11]:
failed_tx = Transaction("tx-flash-victim2", "Flash-loan against Victim2")
network.broadcast(failed_tx, origin="SketchyGuy-Node")
included_failed, failed_block = network.include(
    "SketchyGuy-Node", failed_tx.tx_id, chain
)
print(
    f"{failed_block.proposer} included {included_failed.tx_id} "
    f"in Block #{failed_block.index} (execution comes next)."
)

failed_pool = AMMPool("amm-victim2.eth", 50.0, 100_000.0)
failed_protocol = LendingProtocol("lending-victim2.eth", failed_pool)
failed_protocol.open_loan("Victim2", 40.0, 12_000.0)
failed_provider = FlashLoanProvider(1_000.0)
failed_before = (
    failed_pool.eth_reserve,
    failed_pool.usd_reserve,
    tuple(sorted(failed_protocol.loans)),
    failed_provider.eth_available,
)
failed_observation: dict[str, float] = {}


def failed_attack_action(amount_eth: float):
    failed_pool.swap_eth_for_usd(amount_eth)
    failed_observation["price"] = failed_pool.spot_price
    failed_observation["ratio"] = failed_protocol.collateral_ratio(
        failed_protocol.loans["Victim2"]
    )
    failed_protocol.liquidate("Victim2")
    raise RuntimeError("Liquidation should have raised first.")


try:
    failed_provider.execute(
        40.0, failed_attack_action, failed_pool, failed_protocol
    )
except PositionNotLiquidatableError:
    print(
        f"Attempted dump moved AMM price to "
        f"${failed_observation['price']:,.2f}/ETH."
    )
    print(
        f"Victim2 ratio after dump: {failed_observation['ratio']:.2f}; "
        "liquidation is rejected."
    )
    print("Attack transaction: REVERTED")

failed_after = (
    failed_pool.eth_reserve,
    failed_pool.usd_reserve,
    tuple(sorted(failed_protocol.loans)),
    failed_provider.eth_available,
)
rollback_verified = failed_before == failed_after
print(f"Rollback complete: {rollback_verified}")

receipts.append(
    TransactionReceipt(
        included_failed,
        failed_block,
        "REVERTED",
        185_000,
        "No state change",
    )
)


SketchyGuy-Node included tx-flash-victim2 in Block #2 (execution comes next).
Attempted dump moved AMM price to $617.28/ETH.
Victim2 ratio after dump: 2.06; liquidation is rejected.
Attack transaction: REVERTED
Rollback complete: True


**Read the result:** the AMM price briefly reached $617.28, but Victim2 still had a 2.06 ratio, so liquidation was rejected. The provider restored the pool, the loan list, and its own balance from the snapshot.

Atomicity stops half-finished state. It cannot stop a fully completed transaction from exploiting a bad price rule — that is what the first attack was. The Victim2 attempt was still included; its receipt will say `REVERTED`.


## 5. Defense: stop asking the puddle

A median of independent reports is not the same observation as a thin AMM's reserves. Dump into a fresh pool exactly as before, but point the lending protocol at the `MedianOracle` from notebook 6.

This dump is a **direct method call**, not a gossiped `Transaction`. We already practised `broadcast` / `include` on the heist. Here the only moving part we care about is the price source, so we skip the waiting room on purpose.

The dump still happens. The pool still looks seasick. The protocol just refuses to take medical advice from the pool. Aggregation helps here because the protocol *stopped reading the AMM*, not because the AMM became honest.

The reports are $2,005, $1,995, and $2,000.

> Pause and predict: after the dump pushes the AMM to about $617/ETH, will the protocol liquidate the original 10 ETH / $12,000 loan if its oracle is still near $2,000?


In [14]:
defense_oracle = MedianOracle(
    [
        PriceReport("exchange-A", 2_005.0),
        PriceReport("exchange-B", 1_995.0),
        PriceReport("reference-feed", 2_000.0),
    ]
)
defense_pool = AMMPool("amm-defense.eth", 50.0, 100_000.0)
defense_protocol = LendingProtocol(
    "lending-defense.eth", defense_pool, price_source=defense_oracle
)
defense_victim_loan = defense_protocol.open_loan("defense-victim", 10.0, 12_000.0)
defense_pool.swap_eth_for_usd(40.0)
defense_ratio = defense_protocol.collateral_ratio(defense_victim_loan)

try:
    defense_protocol.liquidate("defense-victim")
except PositionNotLiquidatableError:
    defense_rejected = True
else:
    defense_rejected = False


def defense_verdict(defense_rejected: bool) -> str:
    return "liquidation rejected" if defense_rejected else "liquidation executed"


print(f"Manipulated AMM spot price: ${defense_pool.spot_price:,.2f}/ETH")
print(
    f"Median oracle price: ${defense_oracle.price:,.2f}/ETH; "
    f"protocol ratio: {defense_ratio:.2f}"
)
print(f"Defense result: {defense_verdict(defense_rejected)}")


Manipulated AMM spot price: $617.28/ETH
Median oracle price: $2,000.00/ETH; protocol ratio: 1.67
Defense result: liquidation rejected


**Read the result:** the AMM really moved to $617.28, but the protocol read the median oracle's $2,000 and rejected the healthy loan. Same median idea as notebook 6, now plugged into the protocol that used to consult the puddle.

Median feeds, time-weighted averages, deeper liquidity, freshness limits, circuit breakers, and conservative parameters are layers of defense, not silver bullets.


## 6. Included is not the same as succeeded

Notebook 5 separated origin, gossip receipt, inclusion, and confirmation. Add **execution status**. Both attack transactions were included — that is why we stamped them *before* `execute`. Only one left a state change. The other burned the teaching equivalent of gas and sat on the chain as a receipt of an attempt.

> Pause and predict: which receipt was included but left no modeled state change?


In [17]:
print("Block | Transaction                      | Status   | Gas used | State effect")
for receipt in receipts:
    print(
        f"{receipt.block.index:>5} | {receipt.transaction.description:<32} "
        f"| {receipt.status:<8} | {receipt.gas_used:>8,} | {receipt.state_effect}"
    )
print("Included does not mean succeeded")
valid, message = chain.is_valid()
print(f"Chain valid? {valid} -- {message}")
print(f"Canonical length: {len(chain.chain)} (genesis + {len(receipts)} inclusions)")


Block | Transaction                      | Status   | Gas used | State effect
    1 | Flash-loan oracle manipulation   | SUCCESS  |  310,000 | Attack state committed
    2 | Flash-loan against Victim2       | REVERTED |  185,000 | No state change
Included does not mean succeeded
Chain valid? True -- Chain is valid.
Canonical length: 3 (genesis + 2 inclusions)


**Read the result:** both payloads used notebook 5's `Network.broadcast` and `Network.include`. The heist committed. The Victim2 attempt reverted, but its receipt is still on the chain. Fork choice does not audit the oracle.


## Takeaways

- **Mechanism:** you can shove a thin pool with capital you already own. **Not a guarantee:** you needed to be rich first, or that two separate transactions will stay unopposed.
- **Mechanism:** a flash loan makes huge capital available until the transaction ends. **Not a guarantee:** the lending rule was a good idea.
- **Mechanism:** a flash-loan attack is a `Transaction`: `broadcast`, then `include`. **Not a guarantee:** a missed gossip still lets you include it, or that inclusion means the heist succeeded.
- **Mechanism:** atomicity prevents half-finished state. **Not a guarantee:** a completed exploit of a bad price source gets rolled back out of fairness.
- **Mechanism:** the same `MedianOracle` from notebook 6 stops this heist if the protocol actually reads it. **Not a guarantee:** aggregation is a silver bullet, or that every contract will choose it.
- **Mechanism:** included is not the same as succeeded. **Not a guarantee:** a valid chain is a solvent protocol.

Next: [8. stablecoins.ipynb](8.%20stablecoins.ipynb) — a token that claims to be a dollar, with a reserve the chain cannot see.


## Sources

Atomic borrow-dump-liquidate-repay is a documented DeFi pattern, not a plot twist:

- Qin, K., Zhou, L., Livshits, B., & Gervais, A. (2021). [Attacking the DeFi Ecosystem with Flash Loans for Fun and Profit](https://arxiv.org/abs/2003.03810). Financial Cryptography. Same-transaction credit, oracle manipulation, and why atomicity does not make a bad price rule safe.
- Aave. [Flash Loans](https://aave.com/docs/developers/flash-loans). The production primitive: liquidity that must be returned before the transaction ends, or the whole call reverts.
- Adams, H., Zinsmeister, N., & Robinson, D. (2020). [Uniswap v2 Core](https://uniswap.org/whitepaper.pdf), flash swaps. Receive the asset first, pay it back in the same transaction — the same atomic suitcase.
